# Multidimensional Scaling (MDS)

This notebook implements the Multidimensional Scaling (MDS) function to project the high-dimensional distance matrix (e.g., calculated using MS-SWD) into a 2D space for visualization, matching the methodology used in `fluvgan_1_training_figure_2(1).ipynb`.

### Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.manifold import MDS
import matplotlib.pyplot as plt

### MDS Function Definition

In [2]:
def compute_mds(distance_matrix, n_components=2, dissimilarity='precomputed', random_state=42):
    """
    Performs Multidimensional Scaling (MDS) on a distance matrix.

    Parameters
    ----------
    distance_matrix : ndarray
        The precomputed distance matrix of shape (n_samples, n_samples).
    n_components : int (default 2)
        The number of dimensions in which to project the data.
    dissimilarity : str (default 'precomputed')
        Type of dissimilarity measure ('euclidean' or 'precomputed').
        For precomputed distance matrices, use 'precomputed'.
    random_state : int (default 42)
        Determines the random number generator for initialization.

    Returns
    -------
    embedding : ndarray of shape (n_samples, n_components)
        Projected 2D coordinates.
    """
    mds = MDS(n_components=n_components, dissimilarity=dissimilarity, random_state=random_state)
    embedding = mds.fit_transform(distance_matrix)
    return embedding

### Load Distance Data & Compute MDS Embedding

In [ ]:
# Path to the precomputed distance matrix
# Note: Modify this path if your distance matrix file is located elsewhere
distance_file_path = '../../outputs/fluvgan_1_training_2_test-msswd_distance.npy'
output_csv_path = '../../outputs/fluvgan_1_training_2_test-msswd_embedding.csv'

if os.path.exists(distance_file_path):
    print(f"Loading distance matrix from {distance_file_path}...")
    distances = np.load(distance_file_path)
    print(f"Distance matrix shape: {distances.shape}")
    
    # If distance matrix contains multiple models (e.g. 3D array of shape [n_models, n_samples, n_samples])
    # select the model to analyze (matching model_id in figure 2)
    model_id = 1
    if len(distances.shape) == 3:
        print(f"Selecting distance slice for Model {model_id}...")
        distances_slice = distances[model_id - 1]
    else:
        distances_slice = distances
        
    print("Computing MDS projection (this may take a few moments)...")
    embedding = compute_mds(distances_slice)
    print("MDS projection completed successfully!")
    print(f"Projected embedding shape: {embedding.shape}")
    
    # Save the embedding to a CSV file
    df_embedding = pd.DataFrame(embedding, columns=['Dimension_1', 'Dimension_2'])
    # Add Model ID column to match original structure
    df_embedding['Model'] = model_id
    
    # Ensure outputs directory exists
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df_embedding.to_csv(output_csv_path, index=False)
    print(f"MDS embedding saved to {output_csv_path}")
else:
    print(f"Distance matrix file not found at: {os.path.abspath(distance_file_path)}")
    print("Please update the distance_file_path variable to point to your .npy file.")

### Plot MDS Embedding

In [ ]:
if 'embedding' in locals():
    plt.figure(figsize=(8, 8))
    plt.scatter(embedding[:, 0], embedding[:, 1], c='#bc5090', edgecolors='white', s=30, alpha=0.8)
    plt.xlabel('Dimension 1', fontsize=10)
    plt.ylabel('Dimension 2', fontsize=10)
    plt.title('2D MDS projection of MS-SWD distance matrix', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()
else:
    print("No embedding calculated to plot. Please load a valid distance matrix first.")